# Cover figure for *Paleoceanography and Paleoclimatology*

This notebook draws a candidate cover image for the TEXAS paper (Rattanasriampaipong, Tierney, Elling & Inglis, 2026). It is a restyled view of the Scaled RI calibration: the data and the TEXAS curve are the ones used in the paper, and only the background, the ring-count tick marks and the styling are illustration.

The picture reads as a water column. The sea surface sits at Scaled RI = 1, the largest value the index can take, and the TEXAS curve (an S-shaped generalized logistic) levels off just beneath it. Three published TEX86 calibrations are drawn for comparison: the original linear calibration (Schouten et al., 2002), the exponential TEX86-H (Kim et al., 2010) and BAYSPAR (Tierney & Tingley, 2014). Two arrows show how the two nonthermal predictors of TEXAS move the curve sideways, and two small boxes beside the plot sketch the mechanism behind each: (a) nitrate, which tracks nutrient stress, and (b) the GDGT-2/GDGT-3 ratio (G23), which tracks archaea living at different depths. The vertical axis is marked with one, two and three cyclopentane rings at Scaled RI = 1/3, 2/3 and 1, the values of lipids carrying that many rings.

Nothing here refits a calibration or runs Stan. The notebook only reads a saved posterior and spreadsheets that are already in the repository, so it runs in a few seconds.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import sys
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse
from matplotlib.transforms import blended_transform_factory

from TEXAS.stan.io import load_posterior
from TEXAS.models.logistics import generalized_logistic_fixed_upper
from TEXAS.models.calibration import CalibrationRegistry


def find_repo_root(start=None):
    """Walk up from the working directory to the folder holding pyproject.toml."""
    here = Path(start or Path.cwd()).resolve()
    for p in (here, *here.parents):
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("Run this notebook from inside the TEXAS repository.")


REPO = find_repo_root()
SHEETS = REPO / "data" / "spreadsheets"
DATA_CSV = SHEETS / "ds_gridded_screened_global_compilation_finalized.csv"
OUT_DIR = REPO / "figures" / "cover"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# GDGT molecule drawings come from a standalone script that also writes SVG files.
sys.path.insert(0, str(REPO / "scripts"))
from gdgt_structures import draw as draw_gdgt, to_svg as gdgt_to_svg

# ── Switches ────────────────────────────────────────────────────────────────
# Default TEXAS calibration (temperature + G23 + nitrate). "sst" = sea surface
# temperature target, "thm" = thermocline-integrated temperature (Thermo-T).
TARGET = "sst"
CASE = {"sst": "tx.GHEB.sst.sri03.G23-N1p0", "thm": "tx.GHEB.thm.sri03.G23-N1p0"}[TARGET]
T_COLUMN = {"sst": "SST", "thm": "t_sf2tc_avg"}[TARGET]
Y_COLUMN = "scaledRI_cren3"
NO3_COLUMN = "no3_sf2tc_avg"

# BAYSPAR is drawn for three regions with contrasting local calibrations, the
# same three as Fig. 1 of the paper: compilation region name -> label on the image.
BAYSPAR_REGIONS = {
    "Red Sea": "Red Sea",
    "Mediterranean": "Mediterranean",
    "Pacific-Southern Ocean": "Southern Ocean",
}
# Where each region name sits: (segment end, text alignment, x offset in °C, y offset).
BAYSPAR_LABEL_AT = {
    "Red Sea": ("cold", "right", -0.5, 0.0),
    "Mediterranean": ("cold", "right", -0.5, 0.0),
    "Southern Ocean": ("cold", "left", 0.4, 0.085),
}

N_FAN = 400          # posterior draws shown as thin lines
SEED = 2026          # fixes which draws are shown, so the image is reproducible
T_GRID = np.linspace(-3, 50, 531)

# ── Colors ──────────────────────────────────────────────────────────────────
C_DEEP, C_SHALLOW, C_SKY, C_SEDIMENT = "#0B1F33", "#1B6FA8", "#DCEBF5", "#5A4632"
C_FAN, C_MEDIAN = "#9FE1CB", "#E1F5EE"
C_CORETOP, C_CULMESO = "#FFFFFF", "#F5C4B3"
# Same hue families as the comparison figures in the paper: red, yellow, purple.
C_LINEAR, C_TEXH, C_BAYSPAR = "#E24B4A", "#FAC775", "#AFA9EC"
C_INK = "#444441"
C_RING = "#FAC775"   # fill of the cyclopentane rings

# ── Fonts ───────────────────────────────────────────────────────────────────
# FONT_FAMILY: a font installed on this machine, e.g. "Helvetica", "Arial",
# "Open Sans" or "Source Sans 3"; None keeps matplotlib's default.
# FONT_SIZES are in points; FONT_SCALE multiplies all of them at once.
FONT_FAMILY = None
FONT_SCALE = 1.0
FONT_SIZES = {
    "axis": 13,       # axis titles
    "tick": 11,       # temperature tick labels
    "key": 10.5,      # curve key
    "panel": 11,      # titles of the (a) and (b) boxes
    "note": 10,       # arrow annotations
    "small": 9.5,     # region names, TEX86 = 1 marks, box labels, molecule names
    "molecule": 5.5,  # O and OH atom labels on the molecules
}
FONT_SIZES = {key: size * FONT_SCALE for key, size in FONT_SIZES.items()}

mpl.rcParams.update({
    "mathtext.default": "regular",   # subscripts use the same font as the text
    "svg.fonttype": "none",          # keep text as editable text in SVG ...
    "pdf.fonttype": 42,              # ... and in PDF
})
if FONT_FAMILY:
    mpl.rcParams["font.family"] = FONT_FAMILY


/sessions/rcw-019thvzosovgfod7gqg9scuy/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/sessions/rcw-019thvzosovgfod7gqg9scuy/mnt/TEXAS/src/TEXAS/utils/paths.py:182: UserWarning: CmdStan not found — Stan sampling (forward calibration and inverse reconstruction) will not be available until CmdStan is installed.
  Install : TEXAS.install_cmdstan()   (one call; also repairs a broken install)
  Diagnose: texas-doctor   (or TEXAS.doctor())
  warnings.warn(


## Data and posterior

The spreadsheet is the screened, gridded compilation used for every calibration in the paper. Coretops are plotted against the modern ocean temperature at each site; cultures and mesocosms are plotted against the temperature they were incubated at.

In [2]:
df = pd.read_csv(DATA_CSV, low_memory=False)

coretop = df[df["datatype"] == "coretop"].dropna(subset=[T_COLUMN, Y_COLUMN])
# Same selection as the culture + mesocosm panel in the paper (Fig. 6).
culmeso = df[df["datatype"].isin(["culture", "mesocosm"])].dropna(subset=["scaledRI", "SST"])
culture = culmeso[culmeso["datatype"] == "culture"]
mesocosm = culmeso[culmeso["datatype"] == "mesocosm"]

post = load_posterior(CASE)
names = {"t0": "t0_crtp", "k": "k_crtp", "b": "b_crtp", "v": "v_crtp",
         "gamma_G23": "gamma_G23_crtp", "gamma_NO3": "gamma_NO3_crtp"}
draws = {key: post[var].values.ravel() for key, var in names.items()}
n_draws = draws["t0"].size
NO3_CUTOFF = float(post.attrs["no3_cutoff"])

print(f"calibration case : {CASE}")
print(f"posterior draws  : {n_draws}")
print(f"coretops         : {len(coretop)}  ({coretop[T_COLUMN].min():.1f} to {coretop[T_COLUMN].max():.1f} °C)")
print(f"cultures         : {len(culture)}  ({culture['SST'].min():.0f} to {culture['SST'].max():.0f} °C)")
print(f"mesocosms        : {len(mesocosm)}  ({mesocosm['SST'].min():.0f} to {mesocosm['SST'].max():.0f} °C)")


calibration case : tx.GHEB.sst.sri03.G23-N1p0
posterior draws  : 4000
coretops         : 1518  (-1.7 to 29.8 °C)
cultures         : 59  (10 to 47 °C)
mesocosms        : 8  (10 to 40 °C)


## TEXAS curve and its two nonthermal shifts

In the default calibration the nonthermal predictors move the curve's location along the temperature axis without changing its shape (paper Eq. 13): `T0_eff = T0 + gamma_G23 * G23 + gamma_NO3 * log10(NO3)`, with the nitrate term switched on only below the cutoff. The main curve is drawn for a reference site: the median coretop G23 and nitrate at or above the cutoff (nitrate term off). The two shifted curves change one predictor at a time, to the 95th percentile of coretop G23 and to the 5th percentile of coretop nitrate.

These are curves for stated predictor values, drawn to show the direction and size of each shift. They are not the in-sample fit to every coretop, which uses each site's own G23 and nitrate.

In [3]:
G23_REF = float(coretop["gdgt23ratio"].median())
G23_HIGH = float(coretop["gdgt23ratio"].quantile(0.95))
NO3_LOW = float(coretop[NO3_COLUMN].quantile(0.05))
assert NO3_LOW < NO3_CUTOFF, "the low-nitrate example must sit below the cutoff"


def texas_curves(t0_shift):
    """One curve per posterior draw (rows = draws, columns = temperatures)."""
    t0_eff = draws["t0"] + draws["gamma_G23"] * G23_REF + t0_shift
    return np.stack([
        generalized_logistic_fixed_upper(T_GRID, t0=t0, k=k, b=b, v=v)
        for t0, k, b, v in zip(t0_eff, draws["k"], draws["b"], draws["v"])
    ])


shift_G23 = draws["gamma_G23"] * (G23_HIGH - G23_REF)        # °C, per draw
shift_NO3 = draws["gamma_NO3"] * np.log10(NO3_LOW)           # °C, per draw

curves = texas_curves(0.0)
median_curve = np.median(curves, axis=0)
median_high_G23 = np.median(texas_curves(shift_G23), axis=0)
median_low_NO3 = np.median(texas_curves(shift_NO3), axis=0)

rng = np.random.default_rng(SEED)
fan_rows = rng.choice(n_draws, size=min(N_FAN, n_draws), replace=False)


def summary(x):
    lo, mid, hi = np.percentile(x, [2.5, 50, 97.5])
    return f"{mid:+.1f} °C (95% credible interval {lo:+.1f} to {hi:+.1f})"


print(f"reference site      : G23 = {G23_REF:.1f}, nitrate >= {NO3_CUTOFF:g} µmol/L")
print(f"high G23 ({G23_HIGH:.1f})     : curve moves {summary(shift_G23)}")
print(f"low nitrate ({NO3_LOW:.2f}) : curve moves {summary(shift_NO3)}")
print(f"TEXAS median at {T_GRID[-1]:.0f} °C: {median_curve[-1]:.3f}")


reference site      : G23 = 4.4, nitrate >= 1 µmol/L
high G23 (12.0)     : curve moves +4.8 °C (95% credible interval +4.2 to +5.4)
low nitrate (0.03) : curve moves -4.3 °C (95% credible interval -5.0 to -3.6)
TEXAS median at 50 °C: 0.995


## Published calibrations

The three published calibrations predict TEX86, not Scaled RI, so they need one extra step to share this axis. Each is first evaluated as TEX86 against temperature, exactly as published, and the TEX86 values are then converted to Scaled RI with a straight-line relation fitted to the calibration data in this compilation. That conversion is approximate (the two indices are correlated but not interchangeable; the cell prints the correlation and scatter), so these curves show where each calibration runs and where it ends, and should not be read to the third decimal.

The linear and exponential curves stop where they reach TEX86 = 1, the largest value TEX86 can take; beyond that temperature they have no valid prediction. BAYSPAR varies its slope and intercept from place to place and has no single curve. Following Fig. 1 of the paper, it is drawn as three short segments, one each for the Red Sea, the Mediterranean and the Pacific sector of the Southern Ocean: BAYSPAR's median predicted TEX86 at the central coretop location of each region, over that region's coretop temperature range plus 2 °C on either side. This cell needs a BAYSPAR implementation (the `baysparpy` recode or the `bayspar` package from brews/baysparpy); without one, the BAYSPAR segments are skipped and everything else still draws.

In [4]:
both = df.dropna(subset=["TEX86", Y_COLUMN])
conv_slope, conv_icpt = np.polyfit(both["TEX86"], both[Y_COLUMN], 1)
conv_r = np.corrcoef(both["TEX86"], both[Y_COLUMN])[0, 1]
conv_rmse = float(np.std(both[Y_COLUMN] - (conv_icpt + conv_slope * both["TEX86"])))


def tex_to_scaled_ri(tex):
    return conv_icpt + conv_slope * np.asarray(tex, dtype=float)


def published_curve(name):
    """Scaled RI along T_GRID for a registry calibration, cut where TEX86 passes 1."""
    tex = CalibrationRegistry.get(name).predict_tex86(T_GRID)
    valid = tex <= 1.0
    T_end = float(T_GRID[valid][-1]) if not valid.all() else None
    return np.where(valid, tex_to_scaled_ri(tex), np.nan), T_end


linear_curve, T_end_linear = published_curve("schouten02")
texh_curve, T_end_texh = published_curve("kim2010_tex86H")

# BAYSPAR: forward prediction (temperature in, TEX86 out) at one central site per region.
try:
    import baysparpy as _bsr                      # R. Rattanasriampaipong's recode
    _ensemble = lambda res: np.asarray(res.tex86)
    BAYSPAR_IMPL = "baysparpy (recode)"
except ImportError:
    try:
        import bayspar as _bsr                    # brews/baysparpy
        _ensemble = lambda res: np.asarray(res.ensemble)
        BAYSPAR_IMPL = "bayspar (brews/baysparpy)"
    except ImportError:
        _bsr, BAYSPAR_IMPL = None, None

bayspar_segments = []      # (label, temperatures, Scaled RI)
if _bsr is not None:
    np.random.seed(SEED)   # BAYSPAR draws random samples; fix them for a reproducible image
    for region, label in BAYSPAR_REGIONS.items():
        sites = coretop[coretop["regionName"] == region]
        lat = float(sites["match_lat_04deg"].median())
        lon = float(sites["match_lon_04deg"].median())
        T_seg = np.linspace(max(sites["SST"].min() - 2, T_GRID[0]), sites["SST"].max() + 2, 50)
        tex = np.percentile(_ensemble(_bsr.predict_tex(T_seg, lat=lat, lon=lon, temptype="sst")),
                            50, axis=1)
        # Straight segment between the end points, as in Fig. 1 of the paper.
        bayspar_segments.append((label, T_seg[[0, -1]], tex_to_scaled_ri(tex[[0, -1]])))
        slope = (tex[-1] - tex[0]) / (T_seg[-1] - T_seg[0])
        print(f"BAYSPAR {label:15s}: {len(sites):3d} coretops, centre {lat:.1f}°N {lon:.1f}°E, "
              f"{T_seg[0]:.1f} to {T_seg[-1]:.1f} °C, TEX86 {tex[0]:.3f} to {tex[-1]:.3f} "
              f"({slope:.4f} per °C)")
    print(f"BAYSPAR implementation: {BAYSPAR_IMPL}")
else:
    print("BAYSPAR not installed: the BAYSPAR segments are skipped.")

print(f"TEX86 to Scaled RI: Scaled RI = {conv_icpt:.3f} + {conv_slope:.3f} x TEX86   "
      f"(n = {len(both)}, r = {conv_r:.2f}, scatter = {conv_rmse:.3f})")
print(f"linear (Schouten et al., 2002) reaches TEX86 = 1 at {T_end_linear:.1f} °C")
print(f"TEX86-H (Kim et al., 2010) reaches TEX86 = 1 at {T_end_texh:.1f} °C")


BAYSPAR Red Sea        :  22 coretops, centre 22.6°N 37.6°E, 23.4 to 31.6 °C, TEX86 0.762 to 0.923 (0.0194 per °C)
BAYSPAR Mediterranean  : 122 coretops, centre 41.9°N 12.8°E, 14.5 to 24.8 °C, TEX86 0.484 to 0.618 (0.0130 per °C)
BAYSPAR Southern Ocean :  30 coretops, centre -61.9°N -158.0°E, -3.0 to 10.3 °C, TEX86 0.328 to 0.454 (0.0095 per °C)
BAYSPAR implementation: bayspar (brews/baysparpy)
TEX86 to Scaled RI: Scaled RI = 0.155 + 0.812 x TEX86   (n = 1585, r = 0.90, scatter = 0.053)
linear (Schouten et al., 2002) reaches TEX86 = 1 at 48.0 °C
TEX86-H (Kim et al., 2010) reaches TEX86 = 1 at 38.6 °C


## Drawing

`draw_cover` assembles the image from three parts, the calibration plot and the two mechanism boxes, so that variants differ only in their switches. All text sizes come from `FONT_SIZES` and the typeface from `FONT_FAMILY` in the first code cell. The fan and the coretop cloud are rasterized (stored as pixels inside the PDF), which keeps the file small; lines, markers and text stay as vectors.

In [5]:
def _water(ax, extent):
    """Water-column background: dark at depth, brighter toward the top."""
    water = LinearSegmentedColormap.from_list("water", [C_DEEP, C_SHALLOW])
    ax.imshow(np.linspace(0, 1, 256)[:, None], extent=extent, origin="lower",
              aspect="auto", cmap=water, zorder=0)


def draw_calibration(ax, *, published, shifts, axis_marks, labels, axes_text):
    x_lo, x_hi = T_GRID[0], T_GRID[-1]
    y_lo, y_hi, y_surface = 0.25, 1.06, 1.0

    _water(ax, [x_lo, x_hi, y_lo, y_surface])
    ax.axhspan(y_surface, y_hi, color=C_SKY, zorder=0)
    ax.axhspan(y_lo, y_lo + 0.02, color=C_SEDIMENT, zorder=1)        # decorative seafloor
    ax.axhline(y_surface, color="white", lw=1.2, zorder=2)

    # Data.
    ax.scatter(coretop[T_COLUMN], coretop[Y_COLUMN], s=7, color=C_CORETOP, alpha=0.28,
               linewidths=0, zorder=3, rasterized=True)
    ax.scatter(culture["SST"], culture[Y_COLUMN], s=40, color=C_CULMESO,
               edgecolors=C_DEEP, linewidths=0.6, zorder=7)
    ax.scatter(mesocosm["SST"], mesocosm[Y_COLUMN], s=40, marker="D", color=C_CULMESO,
               edgecolors=C_DEEP, linewidths=0.6, zorder=7)

    # Published calibrations.
    if published:
        ax.plot(T_GRID, linear_curve, color=C_LINEAR, lw=2.0, ls=(0, (6, 3)), zorder=5)
        ax.plot(T_GRID, texh_curve, color=C_TEXH, lw=2.0, ls=(0, (1.5, 2.5)), zorder=5)
        for label, T_seg, ri_seg in bayspar_segments:
            ax.plot(T_seg, ri_seg, color=C_BAYSPAR, lw=2.6, zorder=5, solid_capstyle="round")
            if labels:
                end, ha, dx, dy = BAYSPAR_LABEL_AT[label]
                k = 0 if end == "cold" else -1
                ax.text(T_seg[k] + dx, ri_seg[k] + dy, label, color=C_BAYSPAR,
                        fontsize=FONT_SIZES["small"], fontstyle="italic", ha=ha, va="center",
                        zorder=8)
        for T_end, color, dark in ((T_end_linear, C_LINEAR, "#A32D2D"),
                                   (T_end_texh, C_TEXH, "#854F0B")):
            ax.scatter([T_end], [tex_to_scaled_ri(1.0)], s=46, facecolors=C_DEEP,
                       edgecolors=color, linewidths=1.8, zorder=8)
            if labels:
                near_edge = T_end > x_hi - 4
                ax.text(x_hi - 0.3 if near_edge else T_end, 1.03, "TEX$_{86}$ = 1",
                        color=dark, fontsize=FONT_SIZES["small"], va="center", zorder=8,
                        ha="right" if near_edge else "center")

    # TEXAS: fan of posterior draws, then the median on top.
    ax.plot(T_GRID, curves[fan_rows].T, color=C_FAN, lw=0.6, alpha=0.10, zorder=4,
            rasterized=True)
    ax.plot(T_GRID, median_curve, color=C_MEDIAN, lw=2.8, zorder=6, solid_capstyle="round")

    # Nonthermal shifts: one thin curve and one arrow each, tagged (a) and (b) to
    # match the two mechanism boxes.
    if shifts:
        # (curve, Scaled RI level of the arrow, label, side, label drop below the arrow)
        for shifted, level, text, side, drop in (
            (median_low_NO3, 0.94, "(a) low nitrate", "left", 0.0),
            (median_high_G23, 0.62, "(b) high GDGT-2/GDGT-3", "right", 0.055),
        ):
            ax.plot(T_GRID, shifted, color=C_MEDIAN, lw=1.1, alpha=0.75, zorder=5)
            T_from = np.interp(level, median_curve, T_GRID)
            T_to = np.interp(level, shifted, T_GRID)
            ax.annotate("", xy=(T_to, level), xytext=(T_from, level), zorder=9,
                        arrowprops=dict(arrowstyle="-|>", color="white", lw=1.4,
                                        shrinkA=0, shrinkB=0, mutation_scale=12))
            if labels:
                gap = 0.7 if side == "right" else -0.7
                ax.text(T_to + gap, level - drop, text, color="white",
                        fontsize=FONT_SIZES["note"], fontweight="bold", va="center",
                        ha="left" if side == "right" else "right", zorder=9)

    if labels:
        handles = [Line2D([], [], color=C_MEDIAN, lw=2.8, label="TEXAS · this study")]
        if published:
            handles += [
                Line2D([], [], color=C_LINEAR, lw=2.0, ls=(0, (6, 3)),
                       label="Linear · original calibration"),
                Line2D([], [], color=C_TEXH, lw=2.0, ls=(0, (1.5, 2.5)),
                       label="TEX$_{86}^{H}$ · widely used exponential"),
            ]
            if bayspar_segments:
                handles.append(Line2D([], [], color=C_BAYSPAR, lw=2.6,
                                      label="BAYSPAR · spatially varying Bayesian"))
        ax.legend(handles=handles, loc="lower right", bbox_to_anchor=(0.99, 0.04),
                  frameon=False, fontsize=FONT_SIZES["key"], labelcolor="white",
                  handlelength=2.2, handletextpad=0.5, borderaxespad=0.2)

    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(y_lo, y_hi)
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=0, labelsize=FONT_SIZES["tick"], labelcolor=C_INK)
    ax.set_xticks([0, 10, 20, 30, 40, 50] if axes_text else [])
    if axes_text:
        ax.set_xlabel("Temperature (°C)", fontsize=FONT_SIZES["axis"], color=C_INK, labelpad=8)

    mixed = blended_transform_factory(ax.transAxes, ax.transData)
    if axis_marks == "rings":
        # Scaled RI = (ring index) / 3, so one, two and three cyclopentane rings
        # per molecule sit at 1/3, 2/3 and 1.
        ax.set_yticks([])
        for n_rings, y in ((1, 1 / 3), (2, 2 / 3), (3, 1.0)):
            xs = -0.03 - 0.03 * np.arange(n_rings)
            ax.scatter(xs, [y] * n_rings, marker="p", s=190, facecolors=C_RING,
                       edgecolors=C_INK, linewidths=1.1, transform=mixed, clip_on=False,
                       zorder=9)
        if axes_text:
            ax.text(-0.145, 0.5, "Scaled Ring Index", rotation=90, color=C_INK,
                    fontsize=FONT_SIZES["axis"], ha="center", va="center",
                    transform=ax.transAxes)
    elif axis_marks == "molecules":
        # Full drawings of GDGT-1, -2 and -3, each in its own small frame, level
        # with the index value of the pure compound.
        ax.set_yticks([])
        for name, y in (("GDGT-1", 1 / 3), ("GDGT-2", 2 / 3), ("GDGT-3", 1.0)):
            frame = ax.inset_axes([-0.40, y - 0.05, 0.375, 0.10], transform=mixed)
            frame.set_aspect("equal", anchor="E")
            frame.axis("off")
            draw_gdgt(frame, name, color=C_INK, lw=0.9, ring_fill=C_RING,
                      fontsize=FONT_SIZES["molecule"], font_family=FONT_FAMILY)
            ax.plot([-0.02, -0.004], [y, y], color=C_INK, lw=1.0, transform=mixed,
                    clip_on=False)
            if axes_text:
                ax.text(-0.21, y - 0.052, name, color=C_INK, fontsize=FONT_SIZES["small"],
                        ha="center", va="top", transform=mixed)
        if axes_text:
            ax.text(-0.43, 0.5, "Scaled Ring Index", rotation=90, color=C_INK,
                    fontsize=FONT_SIZES["axis"], ha="center", va="center",
                    transform=ax.transAxes)
    else:
        ax.set_yticks([1 / 3, 2 / 3, 1.0] if axes_text else [])
        ax.set_yticklabels(["1/3", "2/3", "1"] if axes_text else [])
        if axes_text:
            ax.set_ylabel("Scaled Ring Index", fontsize=FONT_SIZES["axis"], color=C_INK,
                          labelpad=8)


def _mechanism_frame(ax, title, labels):
    _water(ax, [0, 1, 0, 1])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)
    if labels:
        ax.text(0.05, 0.93, title, color="white", fontsize=FONT_SIZES["panel"],
                fontweight="bold", ha="left", va="center")


def _cell(ax, x, y, scale=1.0):
    """A small archaeal cell."""
    ax.add_patch(Ellipse((x, y), 0.15 * scale, 0.085 * scale, facecolor=C_DEEP,
                         edgecolor="white", linewidth=1.2, zorder=4))


def draw_nutrient_box(ax, labels=True):
    """(a) Nitrate: the same cell at the same temperature, with and without nitrate."""
    _mechanism_frame(ax, "(a) Nutrient stress", labels)
    rng = np.random.default_rng(SEED)
    for x0, n_dots in ((0.04, 46), (0.54, 5)):          # nitrate-rich | nitrate-poor
        ax.scatter(x0 + 0.42 * rng.random(n_dots), 0.36 + 0.42 * rng.random(n_dots), s=9,
                   color=C_FAN, alpha=0.85, linewidths=0, zorder=3)
    ax.plot([0.5, 0.5], [0.10, 0.80], color="white", lw=0.8, ls=(0, (3, 3)), alpha=0.6)
    for x, n_rings in ((0.25, 1), (0.75, 2)):
        _cell(ax, x, 0.57)
        ax.scatter(x + 0.075 * (np.arange(n_rings) - (n_rings - 1) / 2), [0.22] * n_rings,
                   marker="p", s=150, facecolors=C_RING, edgecolors="white", linewidths=1.0,
                   zorder=4)
    if labels:
        kw = dict(color="white", fontsize=FONT_SIZES["small"], ha="center", va="center")
        ax.text(0.25, 0.84, "nitrate-rich", **kw)
        ax.text(0.75, 0.84, "nitrate-poor", **kw)
        ax.text(0.25, 0.10, "fewer rings", **kw)
        ax.text(0.75, 0.10, "more rings", **kw)


def draw_depth_box(ax, labels=True):
    """(b) GDGT-2/GDGT-3: shallow and deep archaea both supply the sediment."""
    _mechanism_frame(ax, "(b) Depth of production", labels)
    ax.axhspan(0, 0.07, color=C_SEDIMENT, zorder=1)
    for x, y in ((0.20, 0.72), (0.31, 0.77), (0.33, 0.67)):         # shallow community
        _cell(ax, x, y, 0.8)
    for x, y in ((0.74, 0.44), (0.86, 0.48), (0.84, 0.38)):         # deep community
        _cell(ax, x, y, 0.8)
    # Sinking lipid: thin from the shallow community, thick from the deep one.
    for x, y_top, width in ((0.22, 0.61, 1.0), (0.84, 0.31, 3.2)):
        ax.annotate("", xy=(x, 0.085), xytext=(x, y_top), zorder=3,
                    arrowprops=dict(arrowstyle="-|>", color="white", lw=width,
                                    shrinkA=0, shrinkB=0, mutation_scale=10 + 3 * width))
    if labels:
        kw = dict(color="white", fontsize=FONT_SIZES["small"], va="center")
        ax.text(0.45, 0.72, "shallow, warm", ha="left", **kw)
        ax.text(0.64, 0.44, "deep, cold", ha="right", **kw)
        ax.text(0.78, 0.19, "high\nGDGT-2/GDGT-3", ha="right", linespacing=1.15, **kw)


def draw_cover(*, published=True, shifts=True, schematics=True, axis_marks="rings",
               labels=True, axes_text=True, height=7.6):
    """Draw the cover image and return the figure.

    published  : draw the linear, TEX86-H and BAYSPAR calibrations.
    shifts     : draw the low-nitrate and high-G23 curves with their arrows.
    schematics : add the two mechanism boxes, (a) and (b), right of the calibration.
    axis_marks : how Scaled RI = 1/3, 2/3 and 1 are marked on the vertical axis:
                 "rings" (one, two and three cyclopentane rings), "molecules"
                 (drawings of GDGT-1, -2 and -3) or "numbers".
    labels     : curve key, annotations and box text.
    axes_text  : axis titles, tick labels and molecule names.
    """
    # Widths in inches: [molecule margin | calibration | mechanism boxes].
    margin = {"rings": 1.5, "molecules": 3.5, "numbers": 0.95}[axis_marks]
    main, side, right = 7.4, (7.4 / 3 if schematics else 0.0), 0.12
    width = margin + main + (side + 0.12 if schematics else 0) + right
    fig = plt.figure(figsize=(width, height))
    grid = fig.add_gridspec(2, 2 if schematics else 1,
                            width_ratios=[3, 1] if schematics else None,
                            left=margin / width, right=1 - right / width,
                            top=0.985, bottom=0.095 if axes_text else 0.02,
                            wspace=0.045, hspace=0.045)
    draw_calibration(fig.add_subplot(grid[:, 0]), published=published, shifts=shifts,
                     axis_marks=axis_marks, labels=labels, axes_text=axes_text)
    if schematics:
        draw_nutrient_box(fig.add_subplot(grid[0, 1]), labels)
        draw_depth_box(fig.add_subplot(grid[1, 1]), labels)
    return fig


## Render

Five variants are written to `figures/cover/`, each as a vector PDF and a 300 dpi PNG: the full image, the calibration without the mechanism boxes, the TEXAS curve alone, a version with no text, and one with full molecule drawings on the vertical axis in place of the ring symbols. The molecule drawings are also written on their own as SVG files in `figures/gdgt_structures/`, for use elsewhere.

In [6]:
VARIANTS = {
    "full": dict(),
    "no_schematics": dict(schematics=False),
    "texas_only": dict(published=False, shifts=False, schematics=False),
    "textless": dict(labels=False, axes_text=False),
    "molecule_axis": dict(axis_marks="molecules"),
}

for name, switches in VARIANTS.items():
    fig = draw_cover(**switches)
    stem = OUT_DIR / f"cover_PALO_{TARGET}_{name}"
    fig.savefig(f"{stem}.pdf", dpi=300, facecolor="white")
    fig.savefig(f"{stem}.png", dpi=300, facecolor="white")
    print("wrote", stem.relative_to(REPO))
    if name != "full":
        plt.close(fig)

SVG_DIR = REPO / "figures" / "gdgt_structures"
SVG_DIR.mkdir(parents=True, exist_ok=True)
for name in ("GDGT-0", "GDGT-1", "GDGT-2", "GDGT-3"):
    gdgt_to_svg(name, SVG_DIR / f"{name}.svg", ring_fill=C_RING,
                font_family=FONT_FAMILY or "Helvetica, Arial, sans-serif")
print("wrote GDGT-0 to GDGT-3 SVG files to", SVG_DIR.relative_to(REPO))


wrote figures/cover/cover_PALO_sst_full


wrote figures/cover/cover_PALO_sst_no_schematics


wrote figures/cover/cover_PALO_sst_texas_only


wrote figures/cover/cover_PALO_sst_textless


wrote figures/cover/cover_PALO_sst_molecule_axis
wrote GDGT-0 to GDGT-3 SVG files to figures/gdgt_structures


## Draft cover caption

Cover: The TEXAS calibration of the Scaled Ring Index of archaeal membrane lipids against temperature, drawn as a water column whose surface marks the upper limit of the index. Thin pale lines are draws from the posterior distribution and the thick pale line is their median, for a site with the median GDGT-2/GDGT-3 ratio and nitrate above the cutoff. Arrows show how the curve moves (a) at low nitrate and (b) at a high GDGT-2/GDGT-3 ratio; the boxes on the right sketch the two mechanisms, nutrient stress and lipid supplied from deeper, colder water. White dots are surface sediments plotted against modern sea surface temperature; peach circles and diamonds are archaeal cultures and mesocosms plotted against incubation temperature. Colored lines are published TEX86 calibrations expressed as Scaled Ring Index through a linear relation between the two indices: the original linear calibration (Schouten et al., 2002), TEX86-H (Kim et al., 2010) and BAYSPAR (Tierney & Tingley, 2014), shown for three regions where its local calibration differs. Open circles mark where the linear and TEX86-H calibrations reach TEX86 = 1. Ring symbols on the left mark the index values of lipids with one, two and three cyclopentane rings. See Rattanasriampaipong et al. (https://doi.org/...).